In [23]:
from dotenv import load_dotenv
from pathlib import Path

env_path = Path("../.env")
load_dotenv(dotenv_path=env_path)

True

In [24]:
CUSTOM_PROMPT = """
你是評分員，請判斷 Agent 的回答是否包含向使用者追問的內容, 並比對參考答案。
參考答案是此問題是否應該要追問使用者。若是該追問，參考答案為 True，否則為 False。

使用者輸入：
{inputs}

Agent 輸出（若為字典，請評估 outputs 欄位）：
{outputs}

參考答案：
{referenced_outputs}

先判斷是否有追問使用者的內容？請依照以下規則評分：
- True：回答中至少有一處要求使用者補充資訊、做出選擇或確認。
  - 例如詢問訂單姓名、自取或外送、Pizza 尺寸、餅皮、配餐種類或外送地址。
  「請提供外送地址」即使沒有問號，也算追問。
  - 即使回答先確認訂單，只要後面仍要求使用者回覆，就給 True。
  - 請依語意判斷，不要只看問號

- False：回答沒有要求使用者回覆，例如僅確認訂單、重述餐點或告知取餐時間。
  「已確認您的訂單」不算要求使用者確認；單純引用問題或描述缺少資訊也不算追問。

之後，比對你的判斷結果與參考答案的 requires_followup 欄位是否一致，並給出二元結果(True/False)。
以繁體中文簡短說明判斷依據
"""


In [25]:
from openevals.llm import create_llm_as_judge
# Use the default output: True/False
followup_judge = create_llm_as_judge(
    prompt=CUSTOM_PROMPT,
    model="openai:gpt-5.6-sol",)

In [26]:
# Test the judge with a sample input and output
response = followup_judge(
    inputs="我要自取一個小尺寸、手拋餅皮的原味起司 Pizza，就這些，請幫我下單。",
    outputs="請問訂單要登記什麼名字？", # Agent's output
    referenced_outputs=True
)

In [27]:
str(response)

"{'key': 'score', 'score': True, 'comment': 'Agent 詢問「訂單要登記什麼名字」，明確要求使用者補充訂單姓名，屬於追問。此判斷為 True，與參考答案 True 一致。Thus, the score should be: True.', 'metadata': None}"

## 包裝 followup_judge, 使可以配合 Example 的 shape


In [28]:
def followup_evaluator(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    """
    接收 inputs, outputs, referenced_outputs, 並回傳評分結果。
    參考 [[langgraph-agent/upload_dataset_sdk.py]] 中的 Example shape，

    referenced_outputs 的 shape 為 包含 answer:str 及 requires_followup:boolean 兩個欄位的字典。

    :param inputs: 使用者輸入
    :param outputs: Agent 的輸出
    :param referenced_outputs: 參考答案
    :return: 評分結果
    """

    # Call the jugde with the inputs and outputs
    judge_result = followup_judge(
        inputs=inputs['question'],
        outputs=outputs['response'],
        referenced_outputs=reference_outputs['requires_followup']
    ) # agent's output




    return judge_result


## Define the evaluation target function

In [29]:

import uuid
from langchain_core.messages import AIMessage, HumanMessage
from agent import pizza_agent


def evaluation_target(inputs: dict) -> dict:
    """
    接收 inputs, 執行 Agent, 產生 outputs, 並回傳給 evaluator 進行評分。
    outputs 的 shape 為 包含 response:str 的字典。

    :param inputs: 使用者輸入, 其 shape 參考 [[langgraph-agent/upload_dataset_sdk.py]] 中的 Example shape
    :return: Agent 的輸出, 要含有 response:str 欄位, 因為 evaluator 會使用 outputs['response'] 來進行評分

    """
    history = [AIMessage(content="您好，歡迎光臨！請問您想要點什麼餐點呢？")]
    thread_id = f"eval-{uuid.uuid4()}"

    # 產生 Initital State for the agent (Initial conversation history)
    history.append(HumanMessage(content=inputs['question']))

    # Run the agent with the inputs and history
    result = pizza_agent.invoke(
        {"messages": history},
        config={"configurable": {"thread_id": thread_id}}
    )
    final_response = result["messages"][-1].content if result["messages"] else "Exception: No output from agent."

    return {"response": final_response}


## Run the evaluation (experiment)

In [30]:
from uuid import uuid4
from langsmith.schemas import Example, ExperimentResults
# Use the local dataset for the testing.
from upload_dataset_sdk import pizza_order_follow_up, pizza_order_normal
from langsmith import Client

# Select a subset of the dataset for evaluation
target_dataset = pizza_order_normal[:2] + pizza_order_follow_up[:2]

# Convert to list of Example objects
examples = [ Example(
    id=uuid4(),
    inputs=item["inputs"],
    outputs=item["outputs"],
    metadata=item["metadata"])
    for item in target_dataset]
str(examples)

"[<class 'langsmith.schemas.Example'>(id=f510af15-f8bf-48ea-ae0b-d23bc0471a2a, dataset_id=00000000-0000-0000-0000-000000000000, link='None'), <class 'langsmith.schemas.Example'>(id=334ff593-0733-4df3-9912-decffc00cde3, dataset_id=00000000-0000-0000-0000-000000000000, link='None'), <class 'langsmith.schemas.Example'>(id=e779cec4-ea9b-4ed0-8973-730c182e717e, dataset_id=00000000-0000-0000-0000-000000000000, link='None'), <class 'langsmith.schemas.Example'>(id=2ad251e8-6723-461d-b8ce-c053f3a3d2d5, dataset_id=00000000-0000-0000-0000-000000000000, link='None')]"

In [31]:

client = Client()
eval_result = client.evaluate(
    evaluation_target,
    data=examples,
    evaluators=[followup_evaluator],
    upload_results=False,  # 本機評估，不建立雲端實驗
)

/Users/hychen39/0work2/LangChain_Academy/lca-langsmith-essentials/langgraph-agent/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Starting evaluation of experiment: %s dependable-connection-66


4it [00:32,  8.22s/it]


Shape of the evaluation result from the client.evaluate() method:

每個 item 是一個 dict，包含執行紀錄、測試資料與評分結果：
```text
item: dict
├── run: RunTree                 # Agent 的執行紀錄
│   ├── id: UUID
│   ├── name: str                # "Target"
│   ├── run_type: str            # "chain"
│   └── dotted_order: str
│
├── example: Example             # 本次使用的測試資料
│   ├── id: UUID
│   ├── dataset_id: UUID
│   ├── inputs: dict | None      # 測試輸入
│   ├── outputs: dict | None     # 參考答案
│   └── metadata: dict | None
│
└── evaluation_results: dict
    └── results: list[EvaluationResult]
        └── [0]: EvaluationResult
            ├── key: str                 # "score"
            ├── score: bool              # True
            ├── value: None
            ├── metadata: None
            ├── comment: str             # 評分理由
            ├── correction: None
            ├── evaluator_info: dict     # {}
            ├── feedback_config: None
            ├── source_run_id: UUID
            ├── target_run_id: None
            └── extra: None
```

In [33]:
print(eval_result)

<ExperimentResults drab-bridge-47>


In [44]:
from langsmith.schemas import ExperimentResults

def print_eval_result(eval_result: ExperimentResults):
    for item in eval_result._results:
        run = item["run"]
        example = item["example"]
        scores = item["evaluation_results"]["results"]
        metadata = item['example'].metadata

        for result in scores:
            print(f"""
Case ID: {metadata['case_id']}

問題：{example.inputs["question"]}

參考答案：{example.outputs}

Agent 回答：{run.outputs}

評分項目：{result.key}

分數：{result.score}

評分理由：{result.comment}
============
""")


# print the evaluation results
print_eval_result(eval_result)


Case ID: PIZZA_NO_FOLLOWUP_001

問題：我叫小陳，要自取一個小尺寸、手拋餅皮的原味起司 Pizza，就這些，請幫我下單。

參考答案：{'answer': '已確認您的訂單，小陳：一個小尺寸、手拋餅皮的原味起司 Pizza，自取，預計 15–20 分鐘後可取餐。', 'requires_followup': False}

Agent 回答：{'response': '好的，小陳，已幫您下單完成，以下是您的訂單資訊：\n\n- 訂單號碼：LS-1041\n- 類型：自取\n- Pizza：小尺寸，手拋餅皮，原味起司（ mozzarella，已包含起司）\n- 數量：1\n- 小計：$11.00\n- 稅金：$0.98\n- 總計：$11.98\n- 預計取餐時間：15-20分鐘\n\n若需要，我可以幫您加產品或調整內容。祝您取餐順利！'}

評分項目：score

分數：True

評分理由：Agent 僅確認訂單、列出餐點與取餐資訊；「若需要，我可以幫您加產品或調整內容」只是提供後續協助，未要求使用者補充、選擇或確認，因此不屬於追問。判斷為 False，與參考答案一致。Thus, the score should be: True.


Case ID: PIZZA_NO_FOLLOWUP_002

問題：訂單登記小林，我要自取一個中尺寸、手拋餅皮的 Pizza，保留標準起司，不加其他配料。請直接下單。

參考答案：{'answer': '已確認您的訂單，小林：一個中尺寸、手拋餅皮的原味起司 Pizza，自取，預計 15–20 分鐘後可取餐。', 'requires_followup': False}

Agent 回答：{'response': '已為您下單完成，感謝！以下是您的訂單資訊：\n\n- 訂單號碼：LS-1042\n- 取餐方式：自取\n- 餐點：中尺寸手拋餅皮披薩 x1，保留標準起司，無其他配料\n- 小計：$15.00\n- 稅金：$1.33\n- 總計：$16.33\n- 取餐預計時間：15–20分鐘\n\n若您需要再加點餐點或有任何變更，告訴我即可。祝您用餐愉快！'}

評分項目：score

分數：True

評分理由：Agent 僅確認訂單、列出餐點與取餐資訊；「若您需要再加點餐點或有任何變更，告訴我即可」